# Salaries & Wages by Industry x Sex

## Dataset Description
Mean and median monthly salary for Malaysian wage and salary recipients, broken down by **industry sector** and **sex**. Produced by DOSM from the annual **Salaries & Wages Survey**.

> Covers only wage and salary recipients — excludes employers, own-account workers, and unpaid family workers.

## Data Source
| Field | Detail |
|---|---|
| **Publisher** | Department of Statistics Malaysia (DOSM) |
| **Survey** | Salaries & Wages Survey Report (annual) |
| **Portal** | https://open.dosm.gov.my (search: "Salaries Wages Industry") |
| **Parquet URL** | `https://storage.googleapis.com/dosm-public-economy/salaries_industry_sex_xs.parquet` |
| **CSV URL** | `https://storage.googleapis.com/dosm-public-economy/salaries_industry_sex.csv` |
| **License** | CC BY 4.0 — DOSM |
| **Coverage** | 2010 – 2021 |

> **Note on 403 error:** The parquet file is hosted on DOSM's Google Cloud Storage bucket. Some networks block `storage.googleapis.com` with a 403. The `load_parquet()` helper below resolves this by using a browser User-Agent. If it still fails, download the CSV directly from the URL above and load it with `pd.read_csv()`.

## Column Descriptions
| Column | Type | Description |
|---|---|---|
| `date` | date | Year of survey |
| `industry` | string | Industry sector (MSIC 2008) — e.g. Manufacturing, Finance, Education, Health |
| `sex` | string | `overall`, `male`, or `female` |
| `employees` | integer | Number of wage recipients surveyed |
| `mean_wage` | float | Mean monthly salary (RM) |
| `median_wage` | float | Median monthly salary (RM) |

## Relevance to EduNilai
Maps field-of-study to industry sector to estimate what Engineering, Finance, Education and Health graduates actually earn, and tracks the gender wage gap 2010–2021.

In [1]:
import requests
import pandas as pd
import io
import os

def load_parquet(url: str) -> pd.DataFrame:
    """
    Downloads parquet using a browser User-Agent.
    Required for storage.googleapis.com URLs which block Python urllib.
    """
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    return pd.read_parquet(io.BytesIO(resp.content))

URL = 'https://storage.googleapis.com/dosm-public-economy/salaries_industry_sex_xs.parquet'
df = load_parquet(URL)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])

df.head()

HTTPError: 403 Client Error: Forbidden for url: https://storage.googleapis.com/dosm-public-economy/salaries_industry_sex_xs.parquet

In [ ]:
print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nUnique industries:", sorted(df['industry'].unique()))
print("\nUnique sex:", df['sex'].unique())
print("\nYear range:", df['date'].dt.year.min(), "–", df['date'].dt.year.max())

In [ ]:
import matplotlib.pyplot as plt

df_2010 = df[df['date'].dt.year >= 2010].copy()
df_overall = df_2010[df_2010['sex'] == 'overall']

latest_year = df_overall['date'].max()
df_latest = df_overall[df_overall['date'] == latest_year].sort_values('mean_wage', ascending=False)

print(f"Latest year: {latest_year.year}")
print(df_latest[['industry', 'mean_wage', 'median_wage']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df_plot = df_latest.sort_values('mean_wage', ascending=True)
ax.barh(df_plot['industry'], df_plot['mean_wage'], color='steelblue')
ax.set_xlabel('Mean Monthly Wage (RM)')
ax.set_title(f'Mean Monthly Wage by Industry ({latest_year.year})')
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('../data/salary', exist_ok=True)
df_2010.to_csv('../data/salary/salaries_industry_sex.csv', index=False, encoding='utf-8')
print("Saved -> ../data/salary/salaries_industry_sex.csv")
print(f"Rows: {len(df_2010)}")